# **Procesamiento del Lenguaje Natural:** Redes Neuronales Recurrentes (RNNs).

**Angel Navia Vázquez, Pablo Martínez Olmos, Vanessa Gómez Verdejo, Emilio Parrado Hernández**

  * 1.1 (January 2026) Revised and updated version

Departamento de Teoría de la Señal y Comunicaciones

**Universidad Carlos III de Madrid**

<img src='http://www.tsc.uc3m.es/~navia/figures/logo_uc3m_foot.jpg' width=400 />




## 1 - Introducción al procesado secuencial

Ya hemos comentado que la **información textual** presenta una serie de características que hace **difícil su vectorización** para ser utilizada como entrada a una red profunda:

- La **longitud** de los textos de entrada puede ser **variable**: Por ejemplo, podemos tener como entrada a un clasificador una frase con pocas palabras o un documento con varias páginas. Utilizar una vectorización de longitud fija no parece adecuada en estos casos.
- La **posición y secuenciación de las palabras** es importante: los métodos que vectorizan documentos contando frecuencias de aparición de palabras eliminan mucha información lingüística. Por ejemplo, estas dos frases tienen un significado muy diferente, y sin embargo el conteo de palabras es equivalente:

<center>

_La comida era muy **mala**, nada **buena**_

_La comida era muy **buena**, nada **mala**_

</center>

- Puede haber **dependencias de largo plazo**, es decir, la información relevante para una parte del texto puede estar relativamente alejada.

Por ejemplo, en la siguiente frase, si miramos únicamente un **contexto corto** se podría interpretar que la palabra que falta es **alemán**:

<center>

_(...) vivo en **Berlín**, hablo un fluido **#######**._
</center>

pero un **contexto largo** nos puede indicar que la palabra más probable es "**castellano**":

<center>

_Nací en **España** y viví allí durante muchos años de mi infancia. Por eso, aunque ahora vivo en **Berlín**, hablo un fluido **#######**._

</center>


En determinadas aplicaciones, también podríamos tener resultados de **longitud variable a la salida**, por ejemplo en sistemas de **traducción** o de **descripción textual de escenas**. En general podemos tener **diferentes situaciones** de procesado de los **tokens de entrada y salida**:

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/one2many.png' width=600 />
</center>

- El caso **"one-to-one"** es el más habitual en redes sencillas, donde el patrón de entrada tiene un tamaño fijo y la salida también está bien acotada. Por ejemplo, **clasificación de textos** usando una parametrización TFIDF a la entrada.

- El caso **"one-to-many"** podría representar un caso de **descripción de escenas en imágenes**, donde la salida es una lista de palabras.  

- El caso **"many-to-one"** cubre los escenarios en los que se procesan numerosos tokens (palabras) de entrada de forma individual para producir una clasificación final.

- Finalmente, en el caso **"many-to-many"** se parte de una secuencia de palabras para obtener otra secuencia de salida, como por ejemplo en los sistemas de **traducción automática**.



Por tanto, una posible solución a estos inconvenientes es trabajar con modelos que **reciban la información secuencialmente**, y vaya procesando el contenido almacenando determinados **estados internos**.

## 2 -  Las Redes Neuronales Recurrentes (RNNs)

Una **red neuronal recurrente** permite procesar datos de longitud variable, por ejemplo, un texto compuesto por un número variable de tokens:

- Procesa los **patrones de entrada** $\mathbf{x}_t$ de forma **secuencial**
- Almacena información en una **representación interna** (estado $\mathbf{h}_t$), que se va actualizando a medida que se procesa la entrada
- Las **salidas** se calculan en función de la entrada y del valor del estado interno

Una posible representación de la red podría ser como sigue:

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/simpleRNN_W_v2.png' width=300 />
</center>

Las **conexiones dirigidas** indican interconexión entre capas mediante multiplicación por matrices de pesos:

<center>

$\mathbf{h}_t = \sigma(\mathbf{W}_1 \mathbf{x}_t + \mathbf{W}_2 \mathbf{h}_{t-1})$

$\hat{\mathbf{y}} = \mathbf{W}_3 \mathbf{h}_t$

</center>

donde $\mathbf{W}_1$, $\mathbf{W}_2$  y $\mathbf{W}_3$ serían las matrices de pesos del modelo y $\sigma(\cdot)$ sería una función de activación. El cálculo final de las salidas también podría ser un modelo más complejo, por ejemplo un MLP que utilice los valores de la capa oculta como entrada, de la forma $\hat{\mathbf{y}} = f_{MLP}(\mathbf{h}_t$).

### 2.1 - Desdoblamiento temporal (unfolding)

Una forma de **visualizar la secuencia de acciones** implicadas consiste en "desenrollar" temporalmente la red ("**unfolding**"), para representar en un **único grafo** las variables implicadas en **diferentes instantes de tiempo**:


<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/RNN_unfold_c.png' width=600 />
</center>


Fijémonos que obtenemos una **red mayor, pero con pesos compartidos**. Es decir, las conexiones desde la entrada a la capa oculta siempre representan $\mathbf{W}_1$, entre capas ocultas $\mathbf{W}_2$ y hacia la salida $\mathbf{W}_3$.

Asimismo, aunque en la figura se muestran todas las variables, dependiendo del escenario de aplicación, podríamos tener la situación en la que se utilizan **una o más entradas**, y se producen **una o más salidas**, tal y como se ha explicado en los escenarios "one-to-many", "many-to-one", etc..



### 2.2 - Optimización del coste

Un paso final necesario es la **definición del coste a optimizar**. Como la salida de la red puede tener múltiples resultados, la función de coste puede depender de todos ellas, como se indica en la siguiente figura:

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/RNN_unfold_cost.png' width=600 />
</center>

De este modo, el coste $\cal{L}$ sería en el caso general ("xxx-to-many") el promedio de los costes en cada instante de tiempo $\cal{L}_t$, para $t=1, 2, ..., T$. Por ejemplo, en el caso "many-to-one", sólo obtendríamos una decisión final, la salida $\hat{\mathbf{y}}_T$, y el coste sólo contendría el término $\cal{L}_T$.

Una vez definida esta **estructura desplegada**, se podría aplicar la técnica de **retro-propagación de errores** ("**error back-propagation**") para ajustar el modelo, simplemente hay que tener en cuenta la **compartición de pesos** a lo largo de las diferentes etapas.

### 2.3 - Uso de RNNs en modo sencillo o modo generativo

Una vez entrenada, **la red puede utilizarse de dos formas**:

- **Modo sencillo (no iterado)**: tras aplicar las entradas secuencialmente, se va actualizando el estado de la red y se obtiene una respuesta final. En el siguiente ejemplo se muestra el "unfolding"  de una red de este tipo:

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/RNN_sentiment_example.png' width=300 />
</center>




- **Modo generativo**: **las sucesivas respuestas se van realimentando a la entrada**, y de este modo se obtiene una respuesta de longitud arbitraria. Así funcionan los modelos **LLMs usados en IA** (aunque utilizan otra arquitectura interna). Por ejemplo, ante la secuencia de palabras "the", "cat", la red de la siguiente figura podría producir una estimación de la siguiente palabra $x_3$ que podría ser "eats", valor que se añade a la entrada de la red:

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/RNN_generativo_1.png' width=400 />
</center>



Con dicha nueva entrada, se calcula de nuevo la salida de la red:

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/RNN_generativo.png' width=500 />
</center>

que en este caso $x_4$ podría ser "mice", de forma que **se va construyendo la frase "The cat eats mice"**, y el proceso podría continuar produciendo nuevas palabras, habitualmente hasta que la red decide producir un **token de "STOP"**.

### 2.4 - El problema del gradiente atenuado ("vanishing gradient")

La principal limitación de las RNN es que no pueden recordar secuencias muy largas y caen en el problema de la **atenuación del gradiente ("vanishing gradient")**, ya que los gradientes van atravesando **sucesivas funciones de activación** al ser retropropagados (recordemos que en EBP se multiplica por la derivada de la función de activación). El problema es especialmente marcado si se usan sigmoides.

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/vanishing_gradient.png' width=400 />
</center>


La **red recurrente básica** que acabamos de describir presenta una **serie de limitaciones**:

- tiene un único modo de operación, **cada nueva entrada siempre modifica el estado oculto**.
- carece de mecanismos directos para **recordar eventos pasados** de largo plazo
- **no es capaz de ignorar observaciones** presentes

Vamos a presentar una nueva arquitectura que pretende resolver dichos inconvenientes.

## 3 - Las redes **Long Short Term Memory (LSTMs)**

Una **versión evolucionada de red recurrente** la constituyen las conocidas como redes **LSTM** ("**Long Short Term Memory**"), desarrolladas por [Hochreiter y Schmidhuber](https://dl.acm.org/doi/10.1162/neco.1997.9.8.1735). Incluyen un **mecanismo de "puertas" ("gates")** que **activan o desactivan** determinadas conexiones, **controlando el flujo de la información a lo largo del tiempo**.


Veamos una **comparativa entre la operación de una RNN**:

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/oper_RNN.png' width=600 />
</center>

Y una **LSTM**:


<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/oper_LSTM.png' width=600 />
</center>


<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/oper_notation.png' width=600 />
</center>



Por tanto, la operación de ambas redes es análoga en términos de entradas y estados, pero **internamente la red LSTM propaga la información de diferente forma**.

### 3.1 - Funcionamiento interno de la red LSTM

Mientras que la **RNN mantiene un único estado $\mathbf{h}_t$, que funciona como la salida de la red**, la **LSTM utiliza, además de $\mathbf{h}_t$, un estado adicional interno $C_t$**.

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/LSTM_v2.png' width=800 />
</center>

Veremos a continuación en mayor detalle la estructura interna y el funcionamiento de esta red:




- $\mathbf{C}_t$ se conoce como **estado de la red ("cell state")** y es una **representación interna de la información** que se propaga de etapa en etapa siguiendo la línea superior. Es la **variable encargada de recordar y olvidar**. Fijémonos que **en el camino de propagación no hay activaciones, lo que permite la propagación de gradientes sin atenuación**.

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/cell_state.png' width=400 />
</center>

La red LSTM será capaz de **añadir o borrar información** almacenada en el estado $\mathbf{C}_t$ mediante la **operación de una serie de "puertas"** ($f_t$, $i_t$):

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/LSTM_state_update_formulas.png' width=700 />
</center>



La salida de estas puertas se calculan en función de los valores de entrada y valores previos de $\mathbf{h}_{t-1}$:

- la primera operación ($f_t$) se conoce como **"puerta de olvido" (forget gate)**, ya que si $f_t$ produce una salida próxima a cero, el estado interno anterior $\mathbf{C}_{t-1}$ es olvidado (ver cálculo de $\mathbf{C}_{t}$ en la figura anterior).

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/forget_gate.png' width=700 />
</center>

- la segunda operación, representada por **"$i_t$"**, se denomina **"puerta de entrada" (input gate)** que decide qué cantidad del nuevo **estado candidato** $\tilde{{C}}_t$ (calculado a partir de las entrada y estado anteriores) se incorpora finalmente al **estado interno de la red** $C_t$.

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/input_gate.png' width=700 />
</center>



- finalmente, la operación indicada por **"$o_t$"** se denomina **"puerta de salida" (output gate)**, que pondera el estado interno $C_t$ para producir la nueva salida $h_t$.

<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/LSTM_output_gate_formulas.png' width=700 />
</center>

**La red LSTM está disponible en Pytorch**, en la siguiente sección veremos su uso.

### 3.2 - Redes LSTM en Pytorch

El **ajuste de todas las matrices de pesos** ($\mathbf{W}_f$, $\mathbf{W}_x$, $\mathbf{W}_h$, $\mathbf{W}_C$) ha de realizarse mediante la **optimización de una función de coste usando datos de entrenamiento**. Vamos a ver cómo utilizar la librería en ``Pytorch``:

**``LSTM_model = nn.LSTM(input_size, hidden_size, num_layers, bias, dropout)``**

donde los principales parámetros son:

- **input_size**:  número de variables en la entrada $x_t$, normalmente el tamaño del embedding, (no la longitud de la secuencia de entrada).

- **hidden_size**:  número de neuronas en el estado oculto $h_t$

- **num_layers**: número de capas recurrentes. Usar más de 1 implica apilar varias capas LSTM (Default: 1)

- **bias**: indica si usar bias o no (Default: True)

- **dropout**. Añade dropout a las capas interiores. (Default: 0)

Veremos un uso práctico de esta red a continuación.

## 4 - Análisis de opinión en frases financieras

En este notebook vamos a comparar las **prestaciones de diferentes tipos de redes para un problema de clasificación de documentos** en un escenario de **análisis de opinión** (**"sentiment analysis"**):

* un modelo que combina **TF-IDF y Regresión Logística**
* un modelo que usa **promedio de word embeddings (doc embedding) y k-NN**
* un modelo basado en **doc embeddings y MLP**
* un modelo basado en **word embeddings y RNN**

En cada caso, vamos a definir una clase que permita tanto entrenar el modelo como predecir la clasificación de nuevos datos textuales. **Almacenaremos dichos modelos y sus prestaciones para su uso posterior**.


Como base de datos vamos a utilizar [Finantial Phrase Bank](https://www.researchgate.net/profile/Pekka-Malo/publication/251231364_FinancialPhraseBank-v10/data/0c96051eee4fb1d56e000000/FinancialPhraseBank-v10.zip) que contiene cerca de 5000 frases extraídas de textos de noticias financieras:


>*This release of the financial phrase bank covers a collection of 4840 sentences. The selected collection of phrases was annotated by 16 people with adequate background knowledge on financial markets. Three of the annotators were researchers and the remaining 13 annotators were master’s students at Aalto University School of Business with majors primarily in finance, accounting, and economics.*
>
>*The objective of the phrase level annotation task was to classify each example sentence into a positive, negative or neutral category by considering only the information explicitly available in the given sentence. Since the study is focused only on financial and economic domains, the annotators were asked to consider the sentences from the view point of an investor only; i.e. whether the news may have positive, negative or neutral influence on the stock price. As a result, sentences which have a sentiment that is not relevant from an economic or financial perspective are considered neutral.*

Vamos a cargar la base de datos y procederemos al pre-procesado. Esta parte y el entrenamiento de los baselines son idénticas al código que vimos en sesiones anteriores.

=================================================================

**Nota:** A fin de ejecutar los modelos con mayor velocidad, conviene **activar la GPU en Google Colab**:

- **Entorno de ejecución** > Cambiar tipo de entorno de ejecución (``Change runtime type``).
- En **Acelerador de hardware** (``Hardware accelerator``), selecciona **``GPU T4``** (NVIDIA Tesla T4, 16 GB de VRAM) y guardar configuración.

=================================================================

In [1]:
# Ejecutamos este código para preparar el contexto.
from IPython.core.display import Image, display
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib import cm

# Configuración de las figuras matplotlib
plt.rcParams['figure.figsize'] = [8, 6]
plt.rcParams.update({'font.size': 12})

/var/folders/ly/xywcpnyj03v4kvnctf9q83_m0000gn/T/ipykernel_96620/1859716750.py:2: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import Image, display


In [ ]:
# Estas librerías pueden dar problemas de dependencias, conviene instalarlas conjuntamente, para que pip resuelva correctamente las dependencias
!pip install --upgrade numpy pandas scipy gensim spacy nltk torch

In [3]:
import numpy as np
import scipy
import pandas as pd
import spacy
import gensim
import nltk
import torch

print(f"NumPy version: {np.__version__}")
print(f"Scipy version: {scipy.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Spacy version: {spacy.__version__}")
print(f"Gensim version: {gensim.__version__}")
print(f"NLTK version: {nltk.__version__}")
print(f"PyTorch version: {nltk.__version__}")

# 2025
# NumPy version: 1.26.4
# Scipy version: 1.13.1
# Pandas version: 2.2.3
# Spacy version: 3.8.4
# Gensim version: 4.3.3
# NLTK version: 3.9.1
# PyTorch version: 2.6.0+cu124

# 2026
# NumPy version: 2.4.2
# Scipy version: 1.17.0
# Pandas version: 3.0.1
# Spacy version: 3.8.11
# Gensim version: 4.4.0
# NLTK version: 3.9.2
# PyTorch version: 3.9.2

NumPy version: 2.2.6
Scipy version: 1.15.3
Pandas version: 2.3.3
Spacy version: 3.8.14
Gensim version: 4.4.0
NLTK version: 3.9.4
PyTorch version: 3.9.4


In [4]:
import torch
print(f"PyTorch version: {torch.__version__}")
# PyTorch version: 2.6.0+cu124
import pickle
print(f"Pickle version:")
print(pickle)
# <module 'pickle' from '/usr/lib/python3.11/pickle.py'>

PyTorch version: 2.11.0
Pickle version:
<module 'pickle' from '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/pickle.py'>


In [7]:

# Set files location
MYDRIVE="/Volumes/Extreme Pro Particion 1TB/ProcesadoLenguajeNatural/11_MaterialComplementario/"

with open(MYDRIVE + "Sentences.txt", "r", encoding="ISO-8859-1") as f:
    lineas = f.readlines()

print("\nSe han recuperado {} líneas de texto, incluyendo las etiquetas.".format(len(lineas)))
print("\n Esta sería una de las frases:\n")
print(lineas[0].split('@')[0])

print("\n y tendría etiqueta:\n")
print(lineas[0].split('@')[1])



Se han recuperado 5717 líneas de texto, incluyendo las etiquetas.

 Esta sería una de las frases:

According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .

 y tendría etiqueta:

neutral



Queremos entrenar clasificadores para **detectar opiniones negativas**, distinguiéndolas de las opiniones positivas o neutras. Codificamos por tanto las etiquetas negativas como "1" y las neutras/positiva como "0".

In [8]:
frases = [l.split('@')[0] for l in lineas]
opiniones = [l.split('@')[1].replace('\n', '') for l in lineas]

def code_opinion(l):
    if l=='negative':
        d = 1
    else:
        d = 0
    return d

etiquetas_bin = np.array([code_opinion(l) for l in opiniones])

print("Hay un total de {} documentos con opinión negativa, de un total de {}.".format(np.sum(etiquetas_bin), etiquetas_bin.shape[0]))

Hay un total de 723 documentos con opinión negativa, de un total de 5717.


In [9]:
df = pd.DataFrame({"Frases":frases,
                  "Opiniones":opiniones, "Etiquetas": etiquetas_bin})

print("Los primeros documentos son positivos o neutros:\n")
df.head(5)

Los primeros documentos son positivos o neutros:



,Frases,Opiniones,Etiquetas
0,"According to Gran , the company has no plans t...",neutral,0
1,With the new production plant the company woul...,positive,0
2,"For the last quarter of 2010 , Componenta 's n...",positive,0
3,"In the third quarter of 2010 , net sales incre...",positive,0
4,Operating profit rose to EUR 13.1 mn from EUR ...,positive,0


In [10]:
print("\nLos últimos documentos son negativos, de ahí la importancia de aleatorizar:\n")
df.tail(5)



Los últimos documentos son negativos, de ahí la importancia de aleatorizar:



,Frases,Opiniones,Etiquetas
5712,Operating result for the 12-month period decre...,negative,1
5713,HELSINKI Thomson Financial - Shares in Cargote...,negative,1
5714,LONDON MarketWatch -- Share prices ended lower...,negative,1
5715,Operating profit fell to EUR 35.4 mn from EUR ...,negative,1
5716,Sales in Finland decreased by 10.5 % in Januar...,negative,1


Vamos a **aleatorizar los datos disponibles** y definir los conjuntos de **entrenamiento, validación y test**.

**Definiremos únicamente los índices de cada conjunto**, de modo que los podamos aplicar a cualquier representación de los datos que podamos implementar.

In [11]:
from sklearn.model_selection import train_test_split

idx_data = np.arange(0, len(frases), 1)

# Separamos train de test (20% para test)
idx_train, idx_test, y_train, y_test = train_test_split(idx_data, etiquetas_bin, test_size=0.2, random_state=0)

# Separamos train de val (20% para val)
idx_train, idx_val, y_train, y_val = train_test_split(idx_train, y_train, test_size=0.2, random_state=0)

print("El conjunto de entrenamiento tiene {} documentos.".format(idx_train.shape[0]))
print("El conjunto de validación tiene {} documentos.".format(idx_val.shape[0]))
print("El conjunto de test tiene {} documentos.".format(idx_test.shape[0]))

El conjunto de entrenamiento tiene 3658 documentos.
El conjunto de validación tiene 915 documentos.
El conjunto de test tiene 1144 documentos.


### 4.1 - Prestaciones de referencia en problemas desbalanceados

Como ya hemos comentado, vamos a **entrenar diferentes modelos de clasificación binaria** para detectar las opiniones negativas, de modo que se clasifique cada oración como **negativa (1) o neutral/positiva (0)**. Para ello utilizaremos diferentes parametrizaciones de los textos, así como diferentes modelos.

Observe que la base de datos está **muy desbalanceada**, lo que provoca que el **porcentaje de acierto "baseline" NO sea el 50%**, como en el caso de unos datos equilibrados.


In [12]:
print("\nHay un total de {} etiquetas positivas para entrenamiento.".format(np.sum(y_train==1)))
print("Hay un total de {} etiquetas negativas para entrenamiento.".format(np.sum(y_train==0)))

print("\nPrestaciones de un clasificador que siempre decide '0' (positiva)")

acc_baseline_train = np.sum(y_train==0)/y_train.shape[0]*100
acc_baseline_val = np.sum(y_val==0)/y_val.shape[0]*100
acc_baseline_test = np.sum(y_test==0)/y_test.shape[0]*100

print(f"\nEl % de acierto en train de un clasificador base que decide siempre '0' es = {acc_baseline_train}.")
print(f"El % de acierto en val de un clasificador base que decide siempre '0' es = {acc_baseline_val}.")
print(f"El % de acierto en test de un clasificador base que decide siempre '0' es = {acc_baseline_test}.")


Hay un total de 477 etiquetas positivas para entrenamiento.
Hay un total de 3181 etiquetas negativas para entrenamiento.

Prestaciones de un clasificador que siempre decide '0' (positiva)

El % de acierto en train de un clasificador base que decide siempre '0' es = 86.96008747949699.
El % de acierto en val de un clasificador base que decide siempre '0' es = 88.41530054644808.
El % de acierto en test de un clasificador base que decide siempre '0' es = 87.76223776223776.


Vemos una vez más, en este ejemplo, que la **"accuracy" es una medida incompleta de prestaciones**, pues aparentemente estamos acertando el 87% de las veces, pero a costa de **no detectar ninguna etiqueta positiva**. En los siguientes experimentos complementaremos la medida de acierto con las **curvas ROC y su área bajo la curva (AUC)**.

### 4.2 - Clasificación TF-IDF + LR

El primer modelo que vamos a evaluar es el que utiliza una **representación  vectorial TFIDF** y un **Regresor Logístico binario** para detectar las opiniones negativas.

A fin de obtener la representación TFIDF aplicamos primero el **filtrado básico del diccionario** para, a continuación, construir el **BoW y el TFIDF**, siempre respetando la división en datos de **entrenamiento, validación y test**. Recordemos que **los datos de test no se pueden utilizar para entrenar el modelo ni elegir ninguno de sus parámetros**, se deben considerar como "datos no disponibles" hasta el momento de evaluar las prestaciones finales.

#### Ejercicio:

Preprocese los datos y construya la parametrización TFIDF. A continuación entrene un modelo de Regresión Logística y presente las prestaciones resultantes.

In [ ]:
# Prestaciones:
accuracy_train_TFIDF_LR = <COMPLETAR>
accuracy_val_TFIDF_LR = <COMPLETAR>
accuracy_test_TFIDF_LR = <COMPLETAR>

fpr_TFIDF_LR, tpr_TFIDF_LR, thresholds_fprtpr_TFIDF_LR = <COMPLETAR>

fig = plt.figure(figsize=(5, 4))
plt.plot(fpr_TFIDF_LR, tpr_TFIDF_LR, lw=2.5,label='TF-IDF + LR ')
plt.legend(loc=7)
plt.grid()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate)')
plt.title('Curva ROC')
plt.show()

area_roc_TFIDF_LR = <COMPLETAR>

print("\nAccuracy train {}%. Accuracy val {}%.  Accuracy test {}%\n".format(accuracy_train_TFIDF_LR*100, accuracy_val_TFIDF_LR*100,  accuracy_test_TFIDF_LR*100))
print(f"El área bajo la curva ROC de TF-IDF + LR es {area_roc_TFIDF_LR}")


#### Ejercicio:

Almacenar estos resultados para su uso posterior. Utilizar un diccionario guardado en un fichero pickle.

In [ ]:
<COMPLETAR>

### 4.3 - Clasificación Doc Embedding (DE) + LR

Vamos a utilizar un **embedding de tamaño 300**, de modo que cada una de las matrices resultantes es de **tamaño "no. documentos" x 300**. La **estandarización de los Doc embeddings** (columnas de matrices DE) puede interesar cuando el ángulo entre los vectores es más importante que su valor absoluto, por ejemplo en la distancia coseno. Si no sabemos si conviene o no estandarizar, lo decidimos por validación cruzada.

#### Ejercicio:

Generar las matrices de embeddings, entrenar el modelo de Regresión Logística, calcular predicciones y obtener las prestaciones del modelo.

In [ ]:
# Generar matrices de embeddings
<COMPLETAR>

# Entrenar modelo de Regresión Logística
<COMPLETAR>

# Calcular predicciones en los conjuntos de train, val y test
y_pred_DE_LR_train = <COMPLETAR>
y_pred_DE_LR_val = <COMPLETAR>
y_pred_DE_LR_test = <COMPLETAR>

# calcular prestaciones
fpr_DE_LR, tpr_DE_LR, thresholds_fprtpr_DE_LR = <COMPLETAR>
area_roc_DE_LR = <COMPLETAR>

fig = plt.figure(figsize=(5, 4))
plt.plot(fpr_TFIDF_LR, tpr_TFIDF_LR, lw=2.5,label='TF-IDF + LR ')
plt.plot(fpr_DE_LR, tpr_DE_LR, lw=2.5, label='DE + LR')
plt.legend(loc=7)
plt.grid()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate)')
plt.title('Curva ROC')
plt.show()

print(f"El área bajo la curva ROC de TF-IDF + LR es {area_roc_TFIDF_LR}")
print(f"El área bajo la curva ROC de Doc-Embeddings + LR es {area_roc_DE_LR}")

#### Ejercicio:

Almacenar estos resultados para su uso posterior. Utilizar un diccionario guardado en un fichero pickle.

In [ ]:
<COMPLETAR>

### 4.4 - Clasificación Doc-Embedding + k-NN

Vamos a sustituir el clasificador LR por un k-NN.


#### Ejercicio:

Entrenar el modelo kNN usando los embeddings como entrada, calcular predicciones y obtener las prestaciones del modelo.

In [ ]:
# Entrenar un kNN usando los Doc Embeddings como entrada.
< COMPLETAR >

# Calcular predicciones para train, val y test
y_pred_DE_kNN_train = <COMPLETAR>
y_pred_DE_kNN_val = <COMPLETAR>
y_pred_DE_kNN_test = <COMPLETAR>

# calcular prestaciones
fpr_DE_kNN, tpr_DE_kNN, thresholds_fprtpr_DE_kNN = <COMPLETAR>
area_roc_DE_kNN = <COMPLETAR>

In [ ]:
fpr_DE_kNN, tpr_DE_kNN, thresholds_fprtpr_DE_kNN = <COMPLETAR>
area_roc_DE_kNN = <COMPLETAR>

fig = plt.figure(figsize=(5, 4))
plt.plot(fpr_TFIDF_LR, tpr_TFIDF_LR, lw=2.5,label='TF-IDF + LR ')
plt.plot(fpr_DE_LR, tpr_DE_LR, lw=2.5, label='DE + LR')
plt.plot(fpr_DE_kNN, tpr_DE_kNN, lw=2.5, label='DE + k-NN')
plt.legend(loc=7)
plt.grid()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate)')
plt.title('Curva ROC')
plt.show()

print(f"El área bajo la curva ROC de TF-IDF + LR es {area_roc_TFIDF_LR}")
print(f"El área bajo la curva ROC de Doc-Embeddings + LR es {area_roc_DE_LR}")
print(f"El área bajo la curva ROC de Doc-Embeddings + kNN es {area_roc_DE_kNN}")

#### Ejercicio:

Almacenar estos resultados para su uso posterior. Utilizar un diccionario guardado en un fichero pickle.

In [ ]:
<COMPLETAR>

### 4.5 - Clasificación Doc Embedding (DE) + MLP

Vamos a implementar una **MLP de 3 capas como clasificador**, usando como **entrada el Doc Embedding** de los datos. Tal y como hemos visto en clase, el diseño de una red neuronal involucra la selección de un gran número de hiperparámetros y la mejor de las soluciones require la validación cruzada de éstos, que es un proceso arduo y costoso computacionalmente.


<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/DE_MLP.png' width=700 />
</center>

#### Ejercicio:

Entrenar el modelo MLP usando los embeddings como entrada, calcular predicciones y obtener las prestaciones del modelo.

In [ ]:
# Entrenar un MLP usando los Doc Embeddings como entrada.
< COMPLETAR >

# Calcular predicciones para train, val y test
y_pred_DE_MLP_train = < COMPLETAR >
y_pred_DE_MLP_val = < COMPLETAR >
y_pred_DE_MLP_test = < COMPLETAR >

# calcular prestaciones
fpr_DE_MLP, tpr_DE_MLP, thresholds_fprtpr_DE_MLP = < COMPLETAR >
area_roc_DE_MLP = < COMPLETAR >

In [ ]:
fig = plt.figure(figsize=(5, 4))
plt.plot(fpr_TFIDF_LR, tpr_TFIDF_LR, lw=2.5,label='TF-IDF + LR ')
plt.plot(fpr_DE_LR, tpr_DE_LR, lw=2.5, label='DE + LR')
plt.plot(fpr_DE_kNN, tpr_DE_kNN, lw=2.5, label='DE + k-NN')
plt.plot(fpr_DE_MLP, tpr_DE_MLP, lw=2.5, label='DE + MLP')
plt.legend(loc=7)
plt.grid()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate)')
plt.title('Curva ROC')
plt.show()

print(f"El área bajo la curva ROC de TF-IDF + LR es {area_roc_TFIDF_LR}")
print(f"El área bajo la curva ROC de Doc-Embeddings + LR es {area_roc_DE_LR}")
print(f"El área bajo la curva ROC de Doc-Embeddings + kNN es {area_roc_DE_kNN}")
print(f"El área bajo la curva ROC de Doc-Embeddings + MLP es {area_roc_DE_MLP}")

#### Ejercicio:

Almacenar estos resultados para su uso posterior. Utilizar un diccionario guardado en un fichero pickle.

In [ ]:
<COMPLETAR>

### 4.6 - Clasificación usando RNN + Regresión Logística

Finalmente, vamos a evaluar si un **procesado secuencial de las palabras** (word embeddings) utilizando una **RNN** permite una **representación del texto más fiel al contenido**, permitiendo una mejora de las prestaciones del clasificador. El clasificador será una **Regresión Logística sobre el último estado de la RNN**.


<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/LSTM_LR.png' width=700 />
</center>

Primero, vamos a **unificar la longitud de las secuencias de word embeddings**. Esto es solo un requisito de entrada para la función RNN de pytorch. Una vez obtengamos la secuencia de estados de la RNN, **utilizaremos el estado resultante tras procesar la última palabra del texto**.

In [ ]:
frases_filtro_spacy_train = [frases_filtro_spacy[i] for i in idx_train]
frases_filtro_spacy_val = [frases_filtro_spacy[i] for i in idx_val]
frases_filtro_spacy_test = [frases_filtro_spacy[i] for i in idx_test]

# Calculamos y almacenamos la longitud de cada texto
longitudes_train = [len(d) for d in frases_filtro_spacy_train]
longitudes_val = [len(d) for d in frases_filtro_spacy_val]
longitudes_test = [len(d) for d in frases_filtro_spacy_test]

# Máxima longitud
longitudes = longitudes_train + longitudes_val + longitudes_test
max_l = np.max(longitudes)
print("La frase más larga tiene {} palabras.".format(max_l))

In [ ]:
# Igualar longitud de las frases añadiendo un token "comodín" (no se utilizará, es para tener tensores con igual dimensión)
token_comodin = nlp('#')

frases_filtro_spacy_misma_longitud_train = [frases_filtro_spacy_train[i] + [token_comodin] * (max_l-longitudes_train[i]) for i in range(len(frases_filtro_spacy_train))]
frases_filtro_spacy_misma_longitud_val = [frases_filtro_spacy_val[i] + [token_comodin] * (max_l-longitudes_val[i]) for i in range(len(frases_filtro_spacy_val))]
frases_filtro_spacy_misma_longitud_test = [frases_filtro_spacy_test[i] + [token_comodin] * (max_l-longitudes_test[i]) for i in range(len(frases_filtro_spacy_test))]

print("Ejemplo de frase con comodines insertados al final:")
print(frases_filtro_spacy_misma_longitud_train[0])


A continuación creamos la Red Neuronal Recurrente de tipo LSTM. A la hora de construir una RNN debemos especificar los siguientes parámetros

* **input_size** - En nuestro caso dimensión de cada word embedding (300)
* **hidden_dim** - La dimensión del estado de la LSTM
* **n_layers** - Número de **LSTMs apiladas**, tal y como se ilustra en la siguiente figura
* **dropout** - Probabilidad de dropout entre capas (sólo si n_layers>1)

Se aconseja ver la documentación oficial de la capa [LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html) para entender todos sus parámetros:

In [ ]:
# Red con procesado recurrente LSTM y LR en la capa de salida
class LSTM_LR(nn.Module):
    def __init__(self, input_size, output_size, hidden_dim, n_layers, prob_dropout=0.5):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.input_size = input_size

        # Capa LSTM
        # batch_first=True significa que la primera dimensión del tensor de entrada indexa datos distintos
        self.rnn = nn.LSTM(input_size, hidden_dim, n_layers, dropout=prob_dropout, batch_first=True)

        # last, fully-connected layer
        self.fc1 = nn.Linear(hidden_dim, output_size)
        self.logsoftmax = nn.LogSoftmax(dim=1)

        # Capa dropout
        self.dropout = nn.Dropout(p=prob_dropout)

    def forward(self, x, lengths, h0=None):

        '''
        - x: textos codificados con word embeddings. Dimensiones (batch_size, seq_length, input_size)
        - lengths: la longitud de cada texto (antes de meter el token de relleno). Se usa para mirar el estado correcto
          para clasificar.

        Sobre las dimensiones de los tensores de entrada ...:

        - Señal de entrada a RNN tiene dimensiones (batch_size, seq_length, input_size)
        - La inicialización del estado de la RNN tiene dimensiones (n_layers, batch_size, hidden_dim).
          Si se usa None, se inicializa con ceros.
        - La salida de la RNN tiene dimensiones (batch_size, seq_length, hidden_size).
          Esta salida es el estado de la RNN a lo largo del tiempo para cada dato

        '''
        batch_size = x.size(0)
        seq_length = x.size(1)

        # Calculamos la salida de la RNN
        # r_out es la secuencia de estados
        r_out, _ = self.rnn(x, h0)

        # Usamos el estado correspondiente a procesar la última palabra, antes de meter el relleno.
        # Con el reshape pasamos a dimensiones (batch_ize, hidden_dim)
        aux=torch.stack([r_out[[d], lengths[d]-1,:] for d in range(batch_size)]).reshape([-1,self.hidden_dim])

        # Clasificamos usando una log-softmax y aplicamos una exponencial final para obtener las probabilidades
        output = self.logsoftmax(self.fc1(self.dropout(aux)))
        output = torch.exp(output)
        return output


Vamos a ilustrar cómo podemos obtener la salida de la red dados nuestros textos. El primer paso es obtener las **secuencias de word embeddings** de entrada para cada uno de los documentos y almacener esa información como un tensor Ndocs x Nwords x Dim_embedding.

In [ ]:
# Almacenamos los WE como listas de listas para usarlos como entrada a la red tras pasarlos a tensores
WE_train = [[w.vector for w in doc] for doc in frases_filtro_spacy_misma_longitud_train]
WE_val = [[w.vector for w in doc] for doc in frases_filtro_spacy_misma_longitud_val]
WE_test = [[w.vector for w in doc] for doc in frases_filtro_spacy_misma_longitud_test]

print("Esta es la dimensión de los tensores resultantes:")
print(torch.Tensor(WE_train).shape)
print(torch.Tensor(WE_val).shape)
print(torch.Tensor(WE_test).shape)

Instanciamos la clase anterior y probamos a obtener algunas salidas (sin entrenar):

In [ ]:
input_size = 300  # Tamaño del embedding
output_size = 1
n_hidden = 50
n_layers = 1
prob_dropout = 0.5

model_WE_LSTM_LR = LSTM_LR(input_size, output_size, n_hidden, n_layers, prob_dropout)
preds = model_WE_LSTM_LR.forward(torch.Tensor(WE_train[0:5]), longitudes_train[0:5]).detach().numpy().ravel()

print(preds)


#### Ejercicio:

Tal y como hicimos anteriormente, extendemos la clase para añadir un método de entrenamiento ``.fit()`` y un método que estime las probabilidades de salida ``.predict_proba()``.

In [ ]:
<COMPLETAR>

class WE_LSTM_LR_with_train ...
<COMPLETAR>

  def fit...
  <COMPLETAR>

  def predict_proba...
  <COMPLETAR>


#### Ejercicio

Definida la red neuronal y los métodos para su entrenamiento, instanciamos la clase y entrenamos. Utilizamos un estado de la LSTM de 100 dimensiones y una capa de salida softmax y calculamos prestaciones como es habitual.

In [ ]:
<COMPLETAR>

In [ ]:
fpr_WE_LSTM_LR, tpr_WE_LSTM_LR, thresholds_fprtpr_WE_LSTM_LR = metrics.roc_curve(y_test, y_pred_WE_LSTM_LR_test, pos_label=1)

fig = plt.figure(figsize=(5, 4))
plt.plot(fpr_TFIDF_LR, tpr_TFIDF_LR, lw=2.5,label='TF-IDF + LR ')
plt.plot(fpr_DE_LR, tpr_DE_LR, lw=2.5, label='DE + LR')
plt.plot(fpr_DE_kNN, tpr_DE_kNN, lw=2.5, label='DE + k-NN')
plt.plot(fpr_DE_MLP, tpr_DE_MLP, lw=2.5, label='DE + MLP')
plt.plot(fpr_WE_LSTM_LR, tpr_WE_LSTM_LR, lw=2.5, label='WE + LSTM + LR')
plt.legend(loc=7)
plt.grid()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate)')
plt.title('Curva ROC')
plt.show()

area_roc_WE_LSTM_LR = metrics.roc_auc_score(y_test, y_pred_WE_LSTM_LR_test)

print(f"El área bajo la curva ROC de TF-IDF + LR es {area_roc_TFIDF_LR}")
print(f"El área bajo la curva ROC de Doc-Embeddings + LR es {area_roc_DE_LR}")
print(f"El área bajo la curva ROC de Doc-Embeddings + kNN es {area_roc_DE_kNN}")
print(f"El área bajo la curva ROC de Doc-Embeddings + MLP es {area_roc_DE_MLP}")
print(f"El área bajo la curva ROC de Word-Embeddings + LSTM + LR es {area_roc_WE_LSTM_LR}")


#### Ejercicio:

Almacenar estos resultados para su uso posterior. Utilizar un diccionario guardado en un fichero pickle.

In [ ]:
<COMPLETAR>

#### Ejercicio:

Almacenar en un fichero pickle los datos preprocesados para su uso en el entrenamiento de modelos en el próximo tema.

**NOTA:** pickle no permite el almacenamiento de tokens spacy.

In [ ]:
< COMPLETAR >

#### Ejercicio de ampliación (opcional):

Revise los modelos utilizados hasta ahora, e identifique los posibles hiperparámetros utilizados en cada caso. Explore las mejores combinaciones de dichos parámetros. Decida cuidadosamente qué conjunto de datos utilizar en cada caso (entrenamiento, validación, test).

In [ ]:
< COMPLETAR >